In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import svm
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn import metrics

In [ ]:
train=pd.read_csv('/content/train (1).csv')

In [ ]:
train.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [ ]:
train.shape

(103904, 25)

In [ ]:
test=pd.read_csv('/content/test.csv')

In [ ]:
test.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,19556,Female,Loyal Customer,52,Business travel,Eco,160,5,4,...,5,5,5,5,2,5,5,50,44.0,satisfied
1,1,90035,Female,Loyal Customer,36,Business travel,Business,2863,1,1,...,4,4,4,4,3,4,5,0,0.0,satisfied
2,2,12360,Male,disloyal Customer,20,Business travel,Eco,192,2,0,...,2,4,1,3,2,2,2,0,0.0,neutral or dissatisfied
3,3,77959,Male,Loyal Customer,44,Business travel,Business,3377,0,0,...,1,1,1,1,3,1,4,0,6.0,satisfied
4,4,36875,Female,Loyal Customer,49,Business travel,Eco,1182,2,3,...,2,2,2,2,4,2,4,0,20.0,satisfied


In [ ]:
test.shape

(25976, 25)

In [ ]:
train.isnull().sum()

,0
Unnamed: 0,0
id,0
Gender,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Inflight wifi service,0
Departure/Arrival time convenient,0


In [ ]:
test.isnull().sum()

,0
Unnamed: 0,0
id,0
Gender,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Inflight wifi service,0
Departure/Arrival time convenient,0


In [ ]:
train=train.dropna()

In [ ]:
train.isnull().sum()

,0
Unnamed: 0,0
id,0
Gender,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Inflight wifi service,0
Departure/Arrival time convenient,0


In [ ]:
test=test.dropna()

In [ ]:
test.isnull().sum()

,0
Unnamed: 0,0
id,0
Gender,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Inflight wifi service,0
Departure/Arrival time convenient,0


In [ ]:
train['Gender'].value_counts()

,count
Gender,
Female,52576
Male,51018


In [ ]:
train.replace({"Gender":{'Male':0,'Female':1}},inplace=True)

In [ ]:
train['Customer Type'].value_counts()


,count
Customer Type,
Loyal Customer,84662
disloyal Customer,18932


In [ ]:
train.replace({"Customer Type":{'Loyal Customer':0,'disloyal Customer':1}},inplace=True)

In [ ]:
train['Type of Travel'].value_counts()

,count
Type of Travel,
Business travel,71465
Personal Travel,32129


In [ ]:
train.replace({"Type of Travel":{'Business travel':0,'Personal Travel':1}},inplace=True)

In [ ]:
train['Class'].value_counts()

,count
Class,
Business,49533
Eco,46593
Eco Plus,7468


In [ ]:
train.replace({"Class":{'Business':0,'Eco':1,'Eco Plus':2}},inplace=True)

In [ ]:
train['satisfaction'].value_counts()


,count
satisfaction,
neutral or dissatisfied,58697
satisfied,44897


In [ ]:
train.replace({"satisfaction":{'neutral or dissatisfied':0,'satisfied':1}},inplace=True)

In [ ]:
train.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,0,0,13,1,2,460,3,4,...,5,4,3,4,4,5,5,25,18.0,0
1,1,5047,0,1,25,0,0,235,3,2,...,1,1,5,3,1,4,1,1,6.0,0
2,2,110028,1,0,26,0,0,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,1
3,3,24026,1,0,25,0,0,562,2,5,...,2,2,5,3,1,4,2,11,9.0,0
4,4,119299,0,0,61,0,0,214,3,3,...,3,3,4,4,3,3,3,0,0.0,1


In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 103594 entries, 0 to 103903
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103594 non-null  int64  
 1   id                                 103594 non-null  int64  
 2   Gender                             103594 non-null  int64  
 3   Customer Type                      103594 non-null  int64  
 4   Age                                103594 non-null  int64  
 5   Type of Travel                     103594 non-null  int64  
 6   Class                              103594 non-null  int64  
 7   Flight Distance                    103594 non-null  int64  
 8   Inflight wifi service              103594 non-null  int64  
 9   Departure/Arrival time convenient  103594 non-null  int64  
 10  Ease of Online booking             103594 non-null  int64  
 11  Gate location                      103594 no

In [ ]:
test['Gender'].value_counts()

,count
Gender,
Female,13127
Male,12766


In [ ]:
test['Customer Type'].value_counts()


,count
Customer Type,
Loyal Customer,21111
disloyal Customer,4782


In [ ]:
test['Type of Travel'].value_counts()

,count
Type of Travel,
Business travel,17980
Personal Travel,7913


In [ ]:
test['Class'].value_counts()

,count
Class,
Business,12457
Eco,11524
Eco Plus,1912


In [ ]:
test.replace({"Customer Type":{'Loyal Customer':0,'disloyal Customer':1}},inplace=True)

In [ ]:
test.replace({"Type of Travel":{'Business travel':0,'Personal Travel':1}},inplace=True)


In [ ]:
test.replace({"satisfaction":{'neutral or dissatisfied':0,'satisfied':1}},inplace=True)

In [ ]:
test.replace({"Class":{'Business':0,'Eco':1,'Eco Plus':2}},inplace=True)

In [ ]:
test.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,19556,Female,0,52,0,1,160,5,4,...,5,5,5,5,2,5,5,50,44.0,1
1,1,90035,Female,0,36,0,0,2863,1,1,...,4,4,4,4,3,4,5,0,0.0,1
2,2,12360,Male,1,20,0,1,192,2,0,...,2,4,1,3,2,2,2,0,0.0,0
3,3,77959,Male,0,44,0,0,3377,0,0,...,1,1,1,1,3,1,4,0,6.0,1
4,4,36875,Female,0,49,0,1,1182,2,3,...,2,2,2,2,4,2,4,0,20.0,1


In [ ]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25893 entries, 0 to 25975
Data columns (total 25 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Unnamed: 0                         25893 non-null  int64  
 1   id                                 25893 non-null  int64  
 2   Gender                             25893 non-null  object 
 3   Customer Type                      25893 non-null  int64  
 4   Age                                25893 non-null  int64  
 5   Type of Travel                     25893 non-null  int64  
 6   Class                              25893 non-null  int64  
 7   Flight Distance                    25893 non-null  int64  
 8   Inflight wifi service              25893 non-null  int64  
 9   Departure/Arrival time convenient  25893 non-null  int64  
 10  Ease of Online booking             25893 non-null  int64  
 11  Gate location                      25893 non-null  int64  


In [ ]:
X=train.drop(columns=['Unnamed: 0','id','satisfaction'],axis=1)
Y=train['satisfaction']

In [ ]:
print(X)

        Gender  Customer Type  Age  Type of Travel  Class  Flight Distance  \
0            0              0   13               1      2              460   
1            0              1   25               0      0              235   
2            1              0   26               0      0             1142   
3            1              0   25               0      0              562   
4            0              0   61               0      0              214   
...        ...            ...  ...             ...    ...              ...   
103899       1              1   23               0      1              192   
103900       0              0   49               0      0             2347   
103901       0              1   30               0      0             1995   
103902       1              1   22               0      1             1000   
103903       0              0   27               0      0             1723   

        Inflight wifi service  Departure/Arrival time convenien

In [ ]:
print(Y)

0         0
1         0
2         1
3         0
4         1
         ..
103899    0
103900    1
103901    0
103902    0
103903    0
Name: satisfaction, Length: 103594, dtype: int64


In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=2)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model=LogisticRegression()

In [ ]:
model.fit(X_train,Y_train)

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
test_data_pred=model.predict(X_test)

In [ ]:
pred_data=model.predict(X_train)

In [ ]:
acc_train=accuracy_score(Y_train,pred_data)

In [ ]:
print(acc_train)

0.7947752639517346


In [ ]:
pred_data_test=model.predict(X_test)
acc_test=accuracy_score(Y_test,pred_data_test)

In [ ]:
print(acc_test)

0.7875862734687967


In [ ]:
input_data=[1,0,36,0,0,2863,1,1,3,1,5,4,5,4,4,4,4,3,4,5,0,0.0	]
print(input_data)

[1, 0, 36, 0, 0, 2863, 1, 1, 3, 1, 5, 4, 5, 4, 4, 4, 4, 3, 4, 5, 0, 0.0]


In [ ]:
input_data_np=np.asarray(input_data)
print(input_data_np)

[1.000e+00 0.000e+00 3.600e+01 0.000e+00 0.000e+00 2.863e+03 1.000e+00
 1.000e+00 3.000e+00 1.000e+00 5.000e+00 4.000e+00 5.000e+00 4.000e+00
 4.000e+00 4.000e+00 4.000e+00 3.000e+00 4.000e+00 5.000e+00 0.000e+00
 0.000e+00]


In [ ]:
input_data_re=input_data_np.reshape(1,-1)

In [ ]:
pred=model.predict(input_data_re)
print(pred)
if pred[0] == 0:
  print("not Satisfied")
else:
  print("Satisfied")

[1]
Satisfied


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [ ]:
import pickle

In [ ]:
filename='trained_model_air.sav'
pickle.dump(model,open(filename,'wb'))

In [ ]:
loaded_model=pickle.load(open('trained_model_air.sav','rb'))

In [ ]:
input_data=[1,0,36,0,0,2863,1,1,3,1,5,4,5,4,4,4,4,3,4,5,0,0.0	]
print(input_data)
input_data_np=np.asarray(input_data)
print(input_data_np)
input_data_re=input_data_np.reshape(1,-1)
pred=model.predict(input_data_re)
print(pred)
if pred[0] == 0:
  print("not Satisfied")
else:
  print("Satisfied")

[1, 0, 36, 0, 0, 2863, 1, 1, 3, 1, 5, 4, 5, 4, 4, 4, 4, 3, 4, 5, 0, 0.0]
[1.000e+00 0.000e+00 3.600e+01 0.000e+00 0.000e+00 2.863e+03 1.000e+00
 1.000e+00 3.000e+00 1.000e+00 5.000e+00 4.000e+00 5.000e+00 4.000e+00
 4.000e+00 4.000e+00 4.000e+00 3.000e+00 4.000e+00 5.000e+00 0.000e+00
 0.000e+00]
[1]
Satisfied


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [ ]:
!pip install fastapi
!pip install uvicorn
!pip install pickle5
!pip install pydantic
!pip install scikit-learn
!pip install requests
!pip install pypi-json
!pip install pyngrok
!pip install nest-asyncio
!pip install ngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.1/132.1 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pickle5: filename=pickle5-0.0.11-cp310-cp310-linux_x86_64.whl size=255317 sha256=1d30b3b0d7b7e216d0dbc2d810ac1389e8a1dbb773b2b56cc7a868bcfb220620
  Stored in directory: /root/.cache/pip/wheels/7d/14/ef/4aab19d27fa8e58772be5c71c16add0426acf9e1f64353235c
Successfully built pickle5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.1/127.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import json
import uvicorn
from pyngrok import ngrok
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio

In [ ]:
!ngrok authtoken 2jzwuoEiQt4N1T3FVU8m1aGirUz_2DjWtNxaPEpzu38PeThN6

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
app = FastAPI()

In [ ]:
origins = ["*"]

app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [ ]:
class model_input(BaseModel):
    Gender:int
    Customer_Type:int
    Age:int
    Type_of_Travel:int
    Class:int
    Flight_Distance:int
    Inflight_wifi_service:int
    Departure_Arrival_time_convenient:int
    Ease_of_Online_booking:int
    Gate_location:int
    Food_and_drink:int
    Online_boarding:int
    Seat_comfort:int
    Inflight_entertainment:int
    On_board_service:int
    Leg_room_service:int
    Baggage_handling:int
    Checkin_service:int
    Inflight_service:int
    Cleanliness:int
    Departure_Delay_in_Minutes:int
    Arrival_Delay_in_Minutes:float

In [ ]:
loaded_model=pickle.load(open('trained_model_air.sav','rb'))

In [ ]:
@app.post('/Airline_prediction')

def dia_pred(input_parameters : model_input):
    input_data=input_parameters.json()
    input_dictionary=json.loads(input_data)

    Gender=input_dictionary['Gender']
    Customer_Type=input_dictionary['Customer_Type']
    Age=input_dictionary['Age']
    Type_of_Travel=input_dictionary['Type_of_Travel']
    Class=input_dictionary['Class']
    Flight_Distance=input_dictionary['Flight_Distance']
    Inflight_wifi_service=input_dictionary['Inflight_wifi_service']
    Departure_Arrival_time_convenient=input_dictionary['Departure_Arrival_time_convenient']
    Ease_of_Online_booking=input_dictionary['Ease_of_Online_booking']
    Gate_location=input_dictionary['Gate_location']
    Food_and_drink=input_dictionary['Food_and_drink']
    Online_boarding=input_dictionary['Online_boarding']
    Seat_comfort=input_dictionary['Seat_comfort']
    Inflight_entertainment=input_dictionary['Inflight_entertainment']
    On_board_service=input_dictionary['On_board_service']
    Leg_room_service=input_dictionary['Leg_room_service']
    Baggage_handling=input_dictionary['Baggage_handling']
    Checkin_service=input_dictionary['Checkin_service']
    Inflight_service=input_dictionary['Inflight_service']
    Cleanliness=input_dictionary['Cleanliness']
    Departure_Delay_in_Minutes=input_dictionary['Departure_Delay_in_Minutes']
    Arrival_Delay_in_Minutes=input_dictionary['Arrival_Delay_in_Minutes']

    input_list=[Gender,Customer_Type,Age,Type_of_Travel,Class,Flight_Distance,Inflight_wifi_service,Departure_Arrival_time_convenient,Ease_of_Online_booking,Gate_location,Food_and_drink,Online_boarding,Seat_comfort,Inflight_entertainment,On_board_service,Leg_room_service,Baggage_handling,Checkin_service,Inflight_service,Cleanliness,Departure_Delay_in_Minutes,Arrival_Delay_in_Minutes]

    prediction=loaded_model.predict([input_list])

    if prediction[0] == 0:
      return 'not Satisfied'
    else:
      return 'Satisfied'

In [ ]:
public_url=ngrok.connect(8000)

In [ ]:
public_url

<NgrokTunnel: "https://88f2-34-81-122-34.ngrok-free.app" -> "http://localhost:8000">

In [ ]:
nest_asyncio.apply()
uvicorn.run(app,port=8000)

INFO:     Started server process [150]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     2402:3a80:1c23:477a:3867:6220:7ca5:bcb9:0 - "POST /Airline_prediction HTTP/1.1" 200 OK


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
INFO:     Shutting down
